# 02c - Chignolin CV Two-Stage Tm Prediction

This notebook keeps the `02` simulator/CV model, trains it normally on the simulator objective, then freezes it and trains a small mutant-level head on cached CV trajectories to predict one scalar `Tm` per mutant.


In [4]:
# Requires: pip install -e .

import itertools
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from course_project.models.cv_tm_predictor import BackboneModel, MetricHead
from course_project.peptide import load_target_table
from course_project.peptide_tm import (
    build_train_val_test_split,
    build_metric_dataset,
    extract_cv_sequences,
    metric_stats_from_train_mutants,
    predict_metric_df,
    prediction_stats,
    release_cuda_memory,
    run_metric_head_epoch,
    run_sim_epoch,
)
from course_project.utils import resolve_device


### Config


In [5]:
device = torch.device(resolve_device('cuda'))
force_stage1 = False
force_stage2 = True
print('device:', device)

# Data
peptide_dir = Path('../data/peptide')
packed_path = peptide_dir / 'hlda_trajectories_compact.pt'
tm_csv = peptide_dir / 'Tm.csv'
mfpt_csv = peptide_dir / 'mfpt_slice_thr0p34_tF0p25_tU0p57.csv'
evalue_csv = peptide_dir / 'hlda_evalues_thr0p34_tF0p25_tU0p57.csv'

num_runs = 1
train_mutant_count = 15
val_mutant_count = 10
exclude_mutants = {}

history = 1
frames_per_traj = 20000
take_every_kth_frame = 10

# Backbone model
hidden_size = 128
CVs = 12
token_sizes = (450, 100, 80, CVs)
heads = 1
transformer_layers = 2
pre_pyramid_layers = 2
linear_cv_decoder = True
dropout = 0.005
batch_size = 256
weight_decay = 0.0
cls_weight = 0.5
time_lag_steps = 0
time_lag_weight = 0

# Stage 1: simulator training
sim_learning_rate = 1e-4
sim_epochs = 60
sim_eval_every = 2
sim_early_stop_patience = 4

# Stage 2: frozen CV -> mutant Tm head
metric_name = 'Tm'
metric_learning_rate = 3e-4
metric_epochs = 120
metric_eval_every = 1
metric_early_stop_patience = 5
metric_batch_mutants = 10
metric_supervision_weight = 2.0
metric_head_hidden_size = 160
metric_head_dropout = 0.0
metric_selection_key = 'val_loss'
plot_label_points = True

base_run_name = 'chignolin_cv_tm_2stage'
target_df = load_target_table(tm_csv, mfpt_csv, evalue_csv)
target_mutants = set(target_df['mutant'].astype(str).tolist())

packed = torch.load(packed_path, weights_only=False)
x_all = packed['x'].float()
time_all = packed['time'].float()
offsets = packed['traj_offsets'].long()
labels = packed['labels'].long()
mutants = list(packed['mutants'])
n_feat = int(x_all.shape[1])

uniq_mutants = sorted({
    m for m in set(mutants)
    if str(m).strip().upper() not in exclude_mutants and str(m) in target_mutants
})
print('packed path:', packed_path)
print('feature count:', n_feat)
print('all usable mutants:', len(uniq_mutants))
print('train mutants:', train_mutant_count, 'val mutants:', val_mutant_count, 'test mutants:', len(uniq_mutants) - train_mutant_count - val_mutant_count)


device: cuda
packed path: ../data/peptide/hlda_trajectories_compact.pt
feature count: 29
all usable mutants: 37
train mutants: 15 val mutants: 10 test mutants: 12


### Model


### Train


In [ ]:
sim_early_stop_patience = globals().get('sim_early_stop_patience', 2)
metric_early_stop_patience = globals().get('metric_early_stop_patience', 4)

base_results_dir = Path('../results') / base_run_name
base_results_dir.mkdir(parents=True, exist_ok=True)
run_dir = base_results_dir / 'run'
run_dir.mkdir(parents=True, exist_ok=True)
print('base results dir:', base_results_dir)
print('run dir:', run_dir)


def build_pooled_feature_df(metric_dataset, metric_name, summary_type='mean'):
    rows = []
    for item in metric_dataset:
        seq = item['cv_seq'].cpu().numpy()
        pooled = seq.mean(axis=0) if summary_type == 'mean' else np.median(seq, axis=0)
        row = {'mutant': item['mutant'], metric_name: float(item['target_value'])}
        for i, value in enumerate(pooled, start=1):
            row[f'cv_{i}'] = float(value)
        rows.append(row)
    return pd.DataFrame(rows)


def fit_linear_coefficients(train_df, feature_cols, target_col):
    X = train_df[feature_cols].to_numpy(dtype=float)
    y = train_df[target_col].to_numpy(dtype=float)
    X_aug = np.column_stack([np.ones(len(X)), X])
    coef, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
    return coef


def predict_linear_df(df, feature_cols, coef, metric_name):
    X = df[feature_cols].to_numpy(dtype=float)
    X_aug = np.column_stack([np.ones(len(X)), X])
    pred = X_aug @ coef
    out = df[['mutant', metric_name]].copy()
    out[f'pred_{metric_name}'] = pred
    return out


def fit_best_single_cv_decoder(train_df, val_df, test_df, metric_name):
    cv_cols = [c for c in train_df.columns if c.startswith('cv_')]
    candidates = []
    for col in cv_cols:
        coef = fit_linear_coefficients(train_df, [col], metric_name)
        train_pred_df = predict_linear_df(train_df, [col], coef, metric_name)
        val_pred_df = predict_linear_df(val_df, [col], coef, metric_name)
        test_pred_df = predict_linear_df(test_df, [col], coef, metric_name)
        candidates.append({
            'decoder': 'single_cv',
            'decoder_detail': col,
            'train_pred_df': train_pred_df,
            'val_pred_df': val_pred_df,
            'test_pred_df': test_pred_df,
            'train_stats': prediction_stats(train_pred_df, metric_name),
            'val_stats': prediction_stats(val_pred_df, metric_name),
            'test_stats': prediction_stats(test_pred_df, metric_name),
        })
    candidates.sort(key=lambda row: (-row['val_stats']['r2'], row['val_stats']['rmse'], row['val_stats']['mae']))
    return candidates[0]


def fit_linear_combo_decoder(train_df, val_df, test_df, metric_name):
    cv_cols = [c for c in train_df.columns if c.startswith('cv_')]
    best_result = None
    for subset_size in range(1, len(cv_cols) + 1):
        for subset in itertools.combinations(cv_cols, subset_size):
            subset = list(subset)
            coef = fit_linear_coefficients(train_df, subset, metric_name)
            train_pred_df = predict_linear_df(train_df, subset, coef, metric_name)
            val_pred_df = predict_linear_df(val_df, subset, coef, metric_name)
            test_pred_df = predict_linear_df(test_df, subset, coef, metric_name)
            result = {
                'decoder': 'linear_combo',
                'decoder_detail': ' + '.join(subset),
                'train_pred_df': train_pred_df,
                'val_pred_df': val_pred_df,
                'test_pred_df': test_pred_df,
                'train_stats': prediction_stats(train_pred_df, metric_name),
                'val_stats': prediction_stats(val_pred_df, metric_name),
                'test_stats': prediction_stats(test_pred_df, metric_name),
            }
            if best_result is None or (result['val_stats']['r2'], -result['val_stats']['rmse'], -result['val_stats']['mae']) > (best_result['val_stats']['r2'], -best_result['val_stats']['rmse'], -best_result['val_stats']['mae']):
                best_result = result
    return best_result


cfg = {
    'history': history,
    'frames_per_traj': frames_per_traj,
    'take_every_kth_frame': take_every_kth_frame,
    'hidden_size': hidden_size,
    'CVs': int(token_sizes[-1]),
    'token_sizes': tuple(token_sizes),
    'heads': heads,
    'transformer_layers': transformer_layers,
    'pre_pyramid_layers': pre_pyramid_layers,
    'linear_cv_decoder': linear_cv_decoder,
    'dropout': dropout,
    'batch_size': batch_size,
    'weight_decay': weight_decay,
    'cls_weight': cls_weight,
    'time_lag_steps': time_lag_steps,
    'time_lag_weight': time_lag_weight,
    'sim_learning_rate': sim_learning_rate,
    'sim_epochs': sim_epochs,
    'sim_eval_every': sim_eval_every,
    'sim_early_stop_patience': sim_early_stop_patience,
    'metric_name': metric_name,
    'metric_learning_rate': metric_learning_rate,
    'metric_epochs': metric_epochs,
    'metric_eval_every': metric_eval_every,
    'metric_early_stop_patience': metric_early_stop_patience,
    'metric_batch_mutants': metric_batch_mutants,
    'metric_supervision_weight': metric_supervision_weight,
    'metric_head_hidden_size': metric_head_hidden_size,
    'metric_head_dropout': metric_head_dropout,
    'metric_selection_key': metric_selection_key,
}
print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items() if k in ['frames_per_traj', 'take_every_kth_frame', 'hidden_size', 'CVs', 'token_sizes', 'transformer_layers', 'pre_pyramid_layers', 'batch_size', 'cls_weight']}, indent=2))

sim_history_csv = run_dir / 'sim_history.csv'
run_summary_csv = run_dir / 'run_summary.csv'
config_json = run_dir / 'config.json'
expected_backbone_ckpts = [run_dir / f'run_{run_idx:02d}_backbone.pt' for run_idx in range(1, num_runs + 1)]
need_stage1 = force_stage1 or not sim_history_csv.exists() or not run_summary_csv.exists() or any(not p.exists() for p in expected_backbone_ckpts)

if need_stage1:
    sim_history_rows = []
    run_rows = []
    for run_idx in range(1, num_runs + 1):
        backbone_ckpt = run_dir / f'run_{run_idx:02d}_backbone.pt'
        run_seed = int(torch.seed() % (2**32 - 1))
        random.seed(run_seed)
        np.random.seed(run_seed)
        torch.manual_seed(run_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(run_seed)

        split_payload = build_train_val_test_split(
            run_seed=run_seed,
            uniq_mutants=uniq_mutants,
            train_mutant_count=train_mutant_count,
            val_mutant_count=val_mutant_count,
            mutants=mutants,
            history=cfg['history'],
            time_lag_steps=cfg['time_lag_steps'],
            frames_per_traj=cfg['frames_per_traj'],
            take_every_kth_frame=cfg['take_every_kth_frame'],
            x_all=x_all,
            time_all=time_all,
            offsets=offsets,
            labels=labels,
            n_feat=n_feat,
        )

        release_cuda_memory()
        backbone = BackboneModel(
            feat_dim=n_feat,
            history=cfg['history'],
            hidden=cfg['hidden_size'],
            token_sizes=cfg['token_sizes'],
            heads=cfg['heads'],
            token_layers=cfg['transformer_layers'],
            dropout=cfg['dropout'],
            pre_pyramid_layers=cfg['pre_pyramid_layers'],
            linear_cv_decoder=cfg['linear_cv_decoder'],
        ).to(device)
        sim_opt = torch.optim.AdamW(backbone.parameters(), lr=cfg['sim_learning_rate'], weight_decay=cfg['weight_decay'])
        train_loader = DataLoader(TensorDataset(split_payload['train_x'], split_payload['train_dv'], split_payload['train_dv_tau'], split_payload['train_cls']), batch_size=cfg['batch_size'], shuffle=True, drop_last=False)
        val_loader = DataLoader(TensorDataset(split_payload['val_x'], split_payload['val_dv'], split_payload['val_dv_tau'], split_payload['val_cls']), batch_size=cfg['batch_size'], shuffle=False, drop_last=False)

        best_sim_loss = float('inf')
        best_sim_epoch = 0
        sim_bad_epochs = 0
        for epoch in range(1, cfg['sim_epochs'] + 1):
            tr = run_sim_epoch(backbone, train_loader, device=device, time_lag_steps=cfg['time_lag_steps'], time_lag_weight=cfg['time_lag_weight'], cls_weight=cfg['cls_weight'], opt=sim_opt)
            va = run_sim_epoch(backbone, val_loader, device=device, time_lag_steps=cfg['time_lag_steps'], time_lag_weight=cfg['time_lag_weight'], cls_weight=cfg['cls_weight'], opt=None)
            sim_history_rows.append({'run_idx': run_idx, 'run_seed': run_seed, 'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}})
            if va['sim_loss'] < best_sim_loss - 1e-8:
                best_sim_loss = va['sim_loss']
                best_sim_epoch = epoch
                sim_bad_epochs = 0
                torch.save({'run_idx': run_idx, 'run_seed': run_seed, 'epoch': epoch, 'model_state_dict': backbone.state_dict()}, backbone_ckpt)
            else:
                sim_bad_epochs += 1
            if epoch == 1 or epoch == cfg['sim_epochs'] or epoch % cfg['sim_eval_every'] == 0:
                print(f'stage1 run {run_idx:02d} epoch {epoch:03d} | train sim={tr["sim_loss"]:.4f} | val sim={va["sim_loss"]:.4f} | best={best_sim_loss:.4f} bad={sim_bad_epochs}/{cfg["sim_early_stop_patience"]}')
            if sim_bad_epochs >= cfg['sim_early_stop_patience']:
                print(f'stage1 run {run_idx:02d} early stop at epoch {epoch:03d} | best epoch={best_sim_epoch:03d} val sim={best_sim_loss:.4f}')
                break

        run_rows.append({
            'run_idx': run_idx,
            'run_seed': run_seed,
            'sim_final_epoch': epoch,
            'best_sim_epoch': best_sim_epoch,
            'train_mutants': len(split_payload['train_mutants']),
            'val_mutants': len(split_payload['val_mutants']),
            'test_mutants': len(split_payload['test_mutants']),
        })
        release_cuda_memory()

    sim_history_df = pd.DataFrame(sim_history_rows)
    run_summary_df = pd.DataFrame(run_rows)
    sim_history_df.to_csv(sim_history_csv, index=False)
    run_summary_df.to_csv(run_summary_csv, index=False)
    config_json.write_text(json.dumps({**cfg, 'token_sizes': list(cfg['token_sizes']), 'val_mutant_count': val_mutant_count}, indent=2))
else:
    sim_history_df = pd.read_csv(sim_history_csv)
    run_summary_df = pd.read_csv(run_summary_csv)

print('stage1 run summary:')
run_summary_df


base results dir: ../results/chignolin_cv_tm_2stage
run dir: ../results/chignolin_cv_tm_2stage/run
{
  "frames_per_traj": 20000,
  "take_every_kth_frame": 10,
  "hidden_size": 128,
  "CVs": 12,
  "token_sizes": [
    450,
    100,
    80,
    12
  ],
  "transformer_layers": 2,
  "pre_pyramid_layers": 2,
  "batch_size": 256,
  "cls_weight": 0.5
}
stage1 run 01 epoch 001 | train sim=1.2205 | val sim=1.1325 | best=1.1325 bad=0/4
stage1 run 01 epoch 002 | train sim=1.0567 | val sim=1.0356 | best=1.0356 bad=0/4
stage1 run 01 epoch 004 | train sim=0.9471 | val sim=0.9660 | best=0.9660 bad=0/4
stage1 run 01 epoch 006 | train sim=0.9177 | val sim=0.9574 | best=0.9574 bad=0/4


In [ ]:
base_results_dir = Path('../results') / base_run_name
run_dir = base_results_dir / 'run'
print('run dir:', run_dir)

cfg = {
    'history': history,
    'frames_per_traj': frames_per_traj,
    'take_every_kth_frame': take_every_kth_frame,
    'hidden_size': hidden_size,
    'CVs': int(token_sizes[-1]),
    'token_sizes': tuple(token_sizes),
    'heads': heads,
    'transformer_layers': transformer_layers,
    'pre_pyramid_layers': pre_pyramid_layers,
    'linear_cv_decoder': linear_cv_decoder,
    'dropout': dropout,
    'batch_size': batch_size,
    'weight_decay': weight_decay,
    'cls_weight': cls_weight,
    'time_lag_steps': time_lag_steps,
    'time_lag_weight': time_lag_weight,
    'metric_name': metric_name,
    'metric_learning_rate': metric_learning_rate,
    'metric_epochs': metric_epochs,
    'metric_eval_every': metric_eval_every,
    'metric_early_stop_patience': metric_early_stop_patience,
    'metric_batch_mutants': metric_batch_mutants,
    'metric_supervision_weight': metric_supervision_weight,
    'metric_head_hidden_size': metric_head_hidden_size,
    'metric_head_dropout': metric_head_dropout,
    'metric_selection_key': metric_selection_key,
}
print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items() if k in ['hidden_size', 'CVs', 'token_sizes', 'metric_learning_rate', 'metric_supervision_weight', 'metric_head_hidden_size', 'metric_head_dropout', 'metric_selection_key']}, indent=2))

sim_history_csv = run_dir / 'sim_history.csv'
metric_history_csv = run_dir / 'metric_history.csv'
run_summary_csv = run_dir / 'run_summary.csv'
pred_summary_csv = run_dir / 'metric_prediction_summary.csv'
expected_head_ckpts = [run_dir / f'run_{run_idx:02d}_metric_head.pt' for run_idx in range(1, num_runs + 1)]
need_stage2 = force_stage2 or not metric_history_csv.exists() or not pred_summary_csv.exists() or any(not p.exists() for p in expected_head_ckpts)

sim_history_df = pd.read_csv(sim_history_csv)
run_summary_df = pd.read_csv(run_summary_csv)
metric_history_rows = []
pred_rows = []
trained_payloads = []

if need_stage2:
    for _, run_row in run_summary_df.iterrows():
        run_idx = int(run_row['run_idx'])
        run_seed = int(run_row['run_seed'])
        backbone_ckpt = run_dir / f'run_{run_idx:02d}_backbone.pt'
        head_ckpt = run_dir / f'run_{run_idx:02d}_metric_head.pt'

        split_payload = build_train_val_test_split(
            run_seed=run_seed,
            uniq_mutants=uniq_mutants,
            train_mutant_count=train_mutant_count,
            val_mutant_count=val_mutant_count,
            mutants=mutants,
            history=cfg['history'],
            time_lag_steps=cfg['time_lag_steps'],
            frames_per_traj=cfg['frames_per_traj'],
            take_every_kth_frame=cfg['take_every_kth_frame'],
            x_all=x_all,
            time_all=time_all,
            offsets=offsets,
            labels=labels,
            n_feat=n_feat,
        )
        metric_stats = metric_stats_from_train_mutants(split_payload['train_mutants'], target_df, cfg['metric_name'])

        release_cuda_memory()
        backbone = BackboneModel(
            feat_dim=n_feat,
            history=cfg['history'],
            hidden=cfg['hidden_size'],
            token_sizes=cfg['token_sizes'],
            heads=cfg['heads'],
            token_layers=cfg['transformer_layers'],
            dropout=cfg['dropout'],
            pre_pyramid_layers=cfg['pre_pyramid_layers'],
            linear_cv_decoder=cfg['linear_cv_decoder'],
        ).to(device)
        backbone.load_state_dict(torch.load(backbone_ckpt, map_location='cpu', weights_only=False)['model_state_dict'])
        backbone.eval()

        train_cv_sequences = extract_cv_sequences(backbone, split_payload['train_x'], split_payload['train_samples_df'], batch_size=cfg['batch_size'], device=device)
        val_cv_sequences = extract_cv_sequences(backbone, split_payload['val_x'], split_payload['val_samples_df'], batch_size=cfg['batch_size'], device=device)
        test_cv_sequences = extract_cv_sequences(backbone, split_payload['test_x'], split_payload['test_samples_df'], batch_size=cfg['batch_size'], device=device)
        train_metric_dataset = build_metric_dataset(train_cv_sequences, metric_stats, target_df, cfg['metric_name'])
        val_metric_dataset = build_metric_dataset(val_cv_sequences, metric_stats, target_df, cfg['metric_name'])
        test_metric_dataset = build_metric_dataset(test_cv_sequences, metric_stats, target_df, cfg['metric_name'])
        train_pooled_df = build_pooled_feature_df(train_metric_dataset, cfg['metric_name'])
        val_pooled_df = build_pooled_feature_df(val_metric_dataset, cfg['metric_name'])
        test_pooled_df = build_pooled_feature_df(test_metric_dataset, cfg['metric_name'])

        release_cuda_memory()
        head = MetricHead(cfg['CVs'], cfg['hidden_size'], cfg['metric_head_dropout'], inner_size=cfg['metric_head_hidden_size']).to(device)
        head_opt = torch.optim.AdamW(head.parameters(), lr=cfg['metric_learning_rate'], weight_decay=cfg['weight_decay'])

        best_metric_score = float('-inf')
        best_metric_loss = float('inf')
        best_metric_epoch = 0
        metric_bad_epochs = 0
        for epoch in range(1, cfg['metric_epochs'] + 1):
            tr = run_metric_head_epoch(head, train_metric_dataset, metric_stats=metric_stats, metric_batch_mutants=cfg['metric_batch_mutants'], metric_supervision_weight=cfg['metric_supervision_weight'], device=device, opt=head_opt)
            va = run_metric_head_epoch(head, val_metric_dataset, metric_stats=metric_stats, metric_batch_mutants=cfg['metric_batch_mutants'], metric_supervision_weight=cfg['metric_supervision_weight'], device=device, opt=None)
            metric_history_rows.append({'run_idx': run_idx, 'run_seed': run_seed, 'decoder': 'mlp_all_cv', 'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}})
            current_score = -float(va['metric_loss'])
            if current_score > best_metric_score + 1e-8:
                best_metric_score = current_score
                best_metric_loss = float(va['metric_loss'])
                best_metric_epoch = epoch
                metric_bad_epochs = 0
                torch.save({'run_idx': run_idx, 'run_seed': run_seed, 'epoch': epoch, 'state_dict': head.state_dict(), 'metric_stats': metric_stats}, head_ckpt)
            else:
                metric_bad_epochs += 1
            if epoch == 1 or epoch == cfg['metric_epochs'] or epoch % cfg['metric_eval_every'] == 0:
                print(f'stage2 run {run_idx:02d} epoch {epoch:03d} | train loss={tr["metric_loss"]:.4f} R2={tr["tm_r2"]:.3f} | val loss={va["metric_loss"]:.4f} R2={va["tm_r2"]:.3f} | best loss={best_metric_loss:.4f} bad={metric_bad_epochs}/{cfg["metric_early_stop_patience"]}')
            if metric_bad_epochs >= cfg['metric_early_stop_patience']:
                print(f'stage2 run {run_idx:02d} early stop at epoch {epoch:03d} | best epoch={best_metric_epoch:03d} val loss={best_metric_loss:.4f}')
                break

        metric_final_epoch = epoch
        head_payload = torch.load(head_ckpt, map_location='cpu', weights_only=False)
        head.load_state_dict(head_payload['state_dict'])
        head.eval()

        decoder_results = []
        decoder_results.append(fit_best_single_cv_decoder(train_pooled_df, val_pooled_df, test_pooled_df, cfg['metric_name']))
        decoder_results.append(fit_linear_combo_decoder(train_pooled_df, val_pooled_df, test_pooled_df, cfg['metric_name']))
        train_pred_df = predict_metric_df(head, train_metric_dataset, metric_stats, cfg['metric_name'], device)
        val_pred_df = predict_metric_df(head, val_metric_dataset, metric_stats, cfg['metric_name'], device)
        test_pred_df = predict_metric_df(head, test_metric_dataset, metric_stats, cfg['metric_name'], device)
        decoder_results.append({
            'decoder': 'mlp_all_cv',
            'decoder_detail': f'hidden={cfg["metric_head_hidden_size"]}',
            'train_pred_df': train_pred_df,
            'val_pred_df': val_pred_df,
            'test_pred_df': test_pred_df,
            'train_stats': prediction_stats(train_pred_df, cfg['metric_name']),
            'val_stats': prediction_stats(val_pred_df, cfg['metric_name']),
            'test_stats': prediction_stats(test_pred_df, cfg['metric_name']),
        })

        best_decoder_result = sorted(decoder_results, key=lambda row: (-row['val_stats']['r2'], row['val_stats']['rmse'], row['val_stats']['mae']))[0]
        run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'metric_final_epoch'] = metric_final_epoch
        run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'best_metric_epoch'] = best_metric_epoch
        run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'best_metric_loss'] = best_metric_loss
        run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'selected_decoder'] = best_decoder_result['decoder']
        run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'selected_decoder_detail'] = best_decoder_result['decoder_detail']

        for result in decoder_results:
            decoder = result['decoder']
            decoder_detail = result['decoder_detail']
            train_pred_df = result['train_pred_df']
            val_pred_df = result['val_pred_df']
            test_pred_df = result['test_pred_df']
            train_pred_df.to_csv(run_dir / f'run_{run_idx:02d}_{decoder}_train_predictions.csv', index=False)
            val_pred_df.to_csv(run_dir / f'run_{run_idx:02d}_{decoder}_val_predictions.csv', index=False)
            test_pred_df.to_csv(run_dir / f'run_{run_idx:02d}_{decoder}_test_predictions.csv', index=False)
            pred_rows.append({
                'run_idx': run_idx,
                'run_seed': run_seed,
                'decoder': decoder,
                'decoder_detail': decoder_detail,
                'metric': cfg['metric_name'],
                **{f'train_{k}': v for k, v in result['train_stats'].items()},
                **{f'val_{k}': v for k, v in result['val_stats'].items()},
                **{f'test_{k}': v for k, v in result['test_stats'].items()},
            })
            trained_payloads.append({'run_idx': run_idx, 'run_seed': run_seed, 'decoder': decoder, 'decoder_detail': decoder_detail, 'train_pred_df': train_pred_df, 'val_pred_df': val_pred_df, 'test_pred_df': test_pred_df})

    metric_history_df = pd.DataFrame(metric_history_rows)
    metric_prediction_summary_df = pd.DataFrame(pred_rows)
    metric_history_df.to_csv(metric_history_csv, index=False)
    run_summary_df.to_csv(run_summary_csv, index=False)
    metric_prediction_summary_df.to_csv(pred_summary_csv, index=False)
else:
    metric_history_df = pd.read_csv(metric_history_csv)
    metric_prediction_summary_df = pd.read_csv(pred_summary_csv)
    trained_payloads = []
    for _, row in metric_prediction_summary_df.iterrows():
        run_idx = int(row['run_idx'])
        decoder = str(row['decoder'])
        train_pred_df = pd.read_csv(run_dir / f'run_{run_idx:02d}_{decoder}_train_predictions.csv')
        val_pred_df = pd.read_csv(run_dir / f'run_{run_idx:02d}_{decoder}_val_predictions.csv')
        test_pred_df = pd.read_csv(run_dir / f'run_{run_idx:02d}_{decoder}_test_predictions.csv')
        run_seed = int(run_summary_df.loc[run_summary_df['run_idx'] == run_idx, 'run_seed'].iloc[0])
        trained_payloads.append({'run_idx': run_idx, 'run_seed': run_seed, 'decoder': decoder, 'decoder_detail': row.get('decoder_detail', ''), 'train_pred_df': train_pred_df, 'val_pred_df': val_pred_df, 'test_pred_df': test_pred_df})

best_metric_prediction_summary_df = metric_prediction_summary_df.sort_values(['val_r2', 'val_rmse', 'val_mae'], ascending=[False, True, True]).groupby('run_idx', as_index=False).head(1).reset_index(drop=True)
best_keys = {(int(row['run_idx']), str(row['decoder'])) for _, row in best_metric_prediction_summary_df.iterrows()}
trained_payloads = [payload for payload in trained_payloads if (int(payload['run_idx']), str(payload['decoder'])) in best_keys]
all_trained_payloads = trained_payloads if force_stage2 is False else trained_payloads
selected_payload = trained_payloads[0]
metric_name = cfg['metric_name']
train_pred_df = selected_payload['train_pred_df']
val_pred_df = selected_payload['val_pred_df']
test_pred_df = selected_payload['test_pred_df']
print('selected decoder rows:')
best_metric_prediction_summary_df.round(3)


### Evaluation


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6.4), squeeze=False)
ax1, ax2 = axes[:, 0]
for run_idx, sub in sim_history_df.groupby('run_idx'):
    sub = sub.sort_values('epoch')
    ax1.plot(sub['epoch'], sub['train_sim_loss'], color='#2b6cb0', lw=1.5, label=f'run {run_idx} train')
    ax1.plot(sub['epoch'], sub['val_sim_loss'], color='#dd6b20', lw=1.5, ls='--', label=f'run {run_idx} val')
ax1.set_title('Stage 1 simulator loss (validation-selected)')
ax1.set_xlabel('epoch')
ax1.grid(alpha=0.2)
ax1.legend(frameon=False, fontsize=8)

for run_idx, sub in metric_history_df.groupby('run_idx'):
    sub = sub.sort_values('epoch')
    ax2.plot(sub['epoch'], sub['train_metric_loss'], color='#2b6cb0', lw=1.5, label=f'run {run_idx} train')
    ax2.plot(sub['epoch'], sub['val_metric_loss'], color='#dd6b20', lw=1.5, ls='--', label=f'run {run_idx} val')
ax2.set_title(f'Stage 2 {metric_name} head loss (validation-selected)')
ax2.set_xlabel('epoch')
ax2.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
# Decoder comparison scatter plots for the selected experiment.
comparison_payloads = sorted(all_trained_payloads, key=lambda p: (p['run_idx'], p['decoder']))
fig, axes = plt.subplots(len(comparison_payloads), 1, figsize=(7.4, 4.8 * len(comparison_payloads)), squeeze=False)
for row_i, payload in enumerate(comparison_payloads):
    ax = axes[row_i, 0]
    train_df = payload['train_pred_df'][['mutant', metric_name, f'pred_{metric_name}']].copy()
    val_df = payload['val_pred_df'][['mutant', metric_name, f'pred_{metric_name}']].copy()
    test_df = payload['test_pred_df'][['mutant', metric_name, f'pred_{metric_name}']].copy()

    ax.scatter(train_df[metric_name], train_df[f'pred_{metric_name}'], s=28, alpha=0.20, color='#94a3b8', label='TRAIN')
    ax.scatter(val_df[metric_name], val_df[f'pred_{metric_name}'], s=44, alpha=0.72, color='#f59e0b', label='VAL')
    ax.scatter(test_df[metric_name], test_df[f'pred_{metric_name}'], s=64, alpha=0.95, color='#2563eb', marker='D', label='TEST')
    if plot_label_points:
        for _, rr in test_df.iterrows():
            ax.text(rr[metric_name], rr[f'pred_{metric_name}'], str(rr['mutant']), fontsize=8, alpha=0.84, color='#1d4ed8')

    combined = pd.concat([
        train_df[[metric_name, f'pred_{metric_name}']],
        val_df[[metric_name, f'pred_{metric_name}']],
        test_df[[metric_name, f'pred_{metric_name}']],
    ], ignore_index=True)
    lo = float(np.nanmin(combined.to_numpy(dtype=float)))
    hi = float(np.nanmax(combined.to_numpy(dtype=float)))
    pad = 0.05 * (hi - lo + 1e-8)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='0.35', lw=1.2, ls='--')
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)

    val_stats = prediction_stats(val_df, metric_name)
    test_stats = prediction_stats(test_df, metric_name)
    title = '\n'.join([
        f'Run {payload["run_idx"]}: {payload["decoder"]}',
        f'{payload["decoder_detail"]} | VAL R²={val_stats["r2"]:.3f}, TEST R²={test_stats["r2"]:.3f}, TEST |r|={abs(test_stats["pearson"]):.3f}, TEST RMSE={test_stats["rmse"]:.3f}',
    ])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(f'true {metric_name}')
    ax.set_ylabel(f'predicted {metric_name}')
    ax.grid(alpha=0.2)
    ax.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
print('all experiments and runs:')
all_runs_df = pd.concat(
    [
        experiment_outputs[name]['metric_prediction_summary_df'].assign(selected=(name == best_experiment_name))
        for name in experiment_outputs
    ],
    ignore_index=True,
).sort_values(['val_r2', 'val_rmse', 'val_mae'], ascending=[False, True, True]).reset_index(drop=True)
display_cols = [
    'selected', 'experiment', 'run_idx', 'decoder', 'decoder_detail', 'metric',
    'train_n', 'train_rmse', 'train_mae', 'train_r2',
    'val_n', 'val_rmse', 'val_mae', 'val_r2', 'val_pearson', 'val_spearman',
    'test_n', 'test_rmse', 'test_mae', 'test_r2', 'test_pearson', 'test_spearman',
]
all_runs_df[display_cols].round(3)
